# Evaluating Multiple LM Outputs (External)

In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
# imports
import json
import os
import pandas as pd
import importlib.util
import sys
from os.path import join
from copy import deepcopy
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    format_features, format_model_info
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion
from stat_genie.blade_pipeline.additions.analysis.fix_code import \
    check_and_fix_code
from blade_bench.utils import get_dataset_info_path, get_dataset_csv_path

In [7]:
# load files
analysis_subdir_path_1 = "analysis1_output"
analysis_subdir_path_2 = "analysis2_output"
analysis_subdir_path_3 = "analysis3_output"

multirun_filename_1 = "multirun_analyses.json"
multirun_filename_2 = "multirun_analyses.json"
multirun_filename_3 = "multirun_analyses.json"

# use both files to get analysis code paths
multirun_path_1 = join(analysis_subdir_path_1, multirun_filename_1)
multirun_path_2 = join(analysis_subdir_path_2, multirun_filename_2)
multirun_path_3 = join(analysis_subdir_path_3, multirun_filename_3)

with open(multirun_path_1, "r") as file:
    multirun_analyses_1 = json.load(file)

with open(multirun_path_2, "r") as file:
    multirun_analyses_2 = json.load(file)
    
with open(multirun_path_3, "r") as file:
    multirun_analyses_3 = json.load(file)

num_analyses_1 = multirun_analyses_1['n']
num_analyses_2 = multirun_analyses_2['n']
num_analyses_3 = multirun_analyses_3['n']

analysis_code_filenames_1 = [f"llm_analysis_{i}.py" for i in range(num_analyses_1)]
analysis_code_filenames_2 = [f"llm_analysis_{i}.py" for i in range(num_analyses_2)]
analysis_code_filenames_3 = [f"llm_analysis_{i}.py" for i in range(num_analyses_3)]

analysis_code_paths_1 = [join(analysis_subdir_path_1, filename)
                         for filename in analysis_code_filenames_1]

analysis_code_paths_2 = [join(analysis_subdir_path_2, filename)
                         for filename in analysis_code_filenames_2]

analysis_code_paths_3 = [join(analysis_subdir_path_3, filename)
                         for filename in analysis_code_filenames_3]

In [8]:
# load dataset info and csv to get task and dataframe
info_path = get_dataset_info_path(multirun_analyses_1["dataset_name"])
data_path = get_dataset_csv_path(multirun_analyses_1["dataset_name"])

with open(info_path, "r") as file:
    info_json = json.load(file)
    
dataset_task = info_json["research_questions"][0]
df = pd.read_csv(data_path)

In [9]:
llm_provider = "openai"
llm_model = "gpt-5-mini"
llm_assistant = llm(provider=llm_provider, model=llm_model)

[2025-12-05 09:29:01.93][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [10]:
features_1 = format_features(multirun_analyses_1, num_analyses_1, llm_assistant)
features_2 = format_features(multirun_analyses_2, num_analyses_2, llm_assistant)
features_3 = format_features(multirun_analyses_3, num_analyses_3, llm_assistant)

[2025-12-05 09:29:03.05][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 09:29:23.72][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  20.67 seconds
[2025-12-05 09:29:23.73][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 09:29:23.93][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 09:29:44.71][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  20.78 seconds
[2025-12-05 09:29:44.72][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 09:29:44.74][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 09:30:30.23][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  45.

In [11]:
model_info_1 = format_model_info(multirun_analyses_1, num_analyses_1, llm_assistant)
model_info_2 = format_model_info(multirun_analyses_2, num_analyses_2, llm_assistant)
model_info_3 = format_model_info(multirun_analyses_3, num_analyses_3, llm_assistant)

[2025-12-05 09:41:21.87][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 09:41:37.22][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  15.35 seconds
[2025-12-05 09:41:37.23][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 09:41:37.34][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 09:41:56.87][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  19.54 seconds
[2025-12-05 09:41:56.88][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 09:41:56.90][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 09:42:08.43][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  11.

In [12]:
conclusions_1 = {}

for i in range(num_analyses_1):
    # read in the txt file as a string
    conclusion_txt = os.path.abspath(os.path.join(analysis_subdir_path_1, f"final_conclusion_{i}.txt"))
    with open(conclusion_txt, "r", encoding="utf-8") as f:
        conclusion_str = f.read()
    conclusions_1[i] = conclusion_str


conclusions_2 = {}

for i in range(num_analyses_2):
    # read in the txt file as a string
    conclusion_txt = os.path.abspath(os.path.join(analysis_subdir_path_1, f"final_conclusion_{i}.txt"))
    with open(conclusion_txt, "r", encoding="utf-8") as f:
        conclusion_str = f.read()
    conclusions_2[i] = conclusion_str
    
conclusions_3 = {}

for i in range(num_analyses_3):
    # read in the txt file as a string
    conclusion_txt = os.path.abspath(os.path.join(analysis_subdir_path_1, f"final_conclusion_{i}.txt"))
    with open(conclusion_txt, "r", encoding="utf-8") as f:
        conclusion_str = f.read()
    conclusions_3[i] = conclusion_str

In [13]:
llm_judge = llm(provider=llm_provider, model=llm_model)
data_head = df.head(10)

[2025-12-05 09:43:31.20][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [14]:
judge_system_prompt = (
    "You are a meticulous research design evaluator. "
    "Your role is to compare two experimental trials methodologically **and interpretively**.\n\n"
    "You will go through the following reasoning plan step-by-step (internally):\n"
    "1. Understand the research question and dataset context.\n"
    "2. Examine independent, control, and response variables for both trials.\n"
    "3. Analyze the model specifications for structural or methodological similarity.\n"
    "4. Focus more on the content, less on the format.\n"
    "5. Assess whether the trials' conclusions are logically consistent given their setups.\n"
    "6. Detect whether either input is None, invalid, erroneous, or incomplete.\n"
    "   - If **one trial** shows errors or missing components but the other is valid, "
    "     impose a **strong penalty** (reduce all category scores by at least 1 point, "
    "     and cap overall similarity at 2).\n"
    "7. Synthesize your evaluation across all components.\n"
    "8. Output a numerical rating for each category.\n\n"
    "DO NOT include your reasoning — only the final dictionary.\n\n"
    "Scoring scale:\n"
    "1 = completely different\n"
    "2 = somewhat different\n"
    "3 = moderately similar\n"
    "4 = very similar\n"
    "5 = almost identical\n\n"
    "Return output **strictly in dictionary format**:\n"
    "{\n"
    "  \"independent_variables\": <number>,\n"
    "  \"control_variables\": <number>,\n"
    "  \"response_variables\": <number>,\n"
    "  \"model_specification\": <number>,\n"
    "  \"conclusions\": <number>,\n"
    "  \"overall_similarity\": <number>\n"
    "}"
)


def make_judge_prompt(task, data_head, featA, featB, modelA, modelB, conclA, conclB):
    return (
        f"Research Question / Context:\n{task}\n\n"
        "Here is a sample of the dataset to understand the structure and variables:\n"
        f"{data_head}\n\n"
        "Compare the two trials methodologically and interpretively based on the provided variables, model specifications, and conclusions.\n\n"
        "==================== TRIAL A ====================\n\n"
        "Independent Variables:\n"
        f"{featA['independent_variables']}\n\n"
        "Control Variables:\n"
        f"{featA.get('control_variables')}\n\n"
        "Response Variables:\n"
        f"{featA['response_variables']}\n\n"
        "Model Specification:\n"
        f"{modelA}\n\n"
        "Conclusion:\n"
        f"{conclA}\n\n"
        "==================== TRIAL B ====================\n\n"
        "Independent Variables:\n"
        f"{featB['independent_variables']}\n\n"
        "Control Variables:\n"
        f"{featB.get('control_variables')}\n\n"
        "Response Variables:\n"
        f"{featB['response_variables']}\n\n"
        "Model Specification:\n"
        f"{modelB}\n\n"
        "Conclusion:\n"
        f"{conclB}\n\n"
        "Now, following your reasoning plan, provide similarity ratings as JSON only."
    )


In [ ]:
### judge results within each group and between each group.
# within each group there should be 3 choose 2 = 3 pairwise comparisons
# between each group there should be 3 x 3 = 9 pairwise comparisons

In [15]:
### begin with within-group performance
within_group = {1: {}, 2: {}, 3: {}}
for i in range(num_analyses_1):
    for j in range(i + 1, num_analyses_1):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_1[i],
            features_1[j],
            multirun_analyses_1['analyses'][str(i)]['m_code'],
            multirun_analyses_1['analyses'][str(j)]['m_code'],
            conclusions_1[i],
            conclusions_1[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        within_group[1][(i, j)] = response_dict
for i in range(num_analyses_2):
    for j in range(i + 1, num_analyses_2):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_2[i],
            features_2[j],
            multirun_analyses_2['analyses'][str(i)]['m_code'],
            multirun_analyses_2['analyses'][str(j)]['m_code'],
            conclusions_2[i],
            conclusions_2[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        within_group[2][(i, j)] = response_dict
for i in range(num_analyses_3):
    for j in range(i + 1, num_analyses_3):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_3[i],
            features_3[j],
            multirun_analyses_3['analyses'][str(i)]['m_code'],
            multirun_analyses_3['analyses'][str(j)]['m_code'],
            conclusions_3[i],
            conclusions_3[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        within_group[3][(i, j)] = response_dict

[2025-12-05 09:46:13.20][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 09:46:34.98][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  21.78 seconds
[2025-12-05 09:46:34.99][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 09:46:35.03][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 09:47:05.38][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  30.35 seconds
[2025-12-05 09:47:05.39][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 09:47:05.43][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 09:47:18.28][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  12.

In [16]:
### now do between-group performance
between_group = { (1, 2): {}, (1, 3): {}, (2, 3): {} }
for i in range(num_analyses_1):
    for j in range(num_analyses_2):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_1[i],
            features_2[j],
            multirun_analyses_1['analyses'][str(i)]['m_code'],
            multirun_analyses_2['analyses'][str(j)]['m_code'],
            conclusions_1[i],
            conclusions_2[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        between_group[(1, 2)][(i, j)] = response_dict
for i in range(num_analyses_1):
    for j in range(num_analyses_3):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_1[i],
            features_3[j],
            multirun_analyses_1['analyses'][str(i)]['m_code'],
            multirun_analyses_3['analyses'][str(j)]['m_code'],
            conclusions_1[i],
            conclusions_3[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        between_group[(1, 3)][(i, j)] = response_dict
for i in range(num_analyses_2):
    for j in range(num_analyses_3):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_2[i],
            features_3[j],
            multirun_analyses_2['analyses'][str(i)]['m_code'],
            multirun_analyses_3['analyses'][str(j)]['m_code'],
            conclusions_2[i],
            conclusions_3[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        between_group[(2, 3)][(i, j)] = response_dict

[2025-12-05 09:49:59.43][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 09:50:14.16][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  14.73 seconds
[2025-12-05 09:50:14.16][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 09:50:14.19][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 09:50:30.32][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  16.13 seconds
[2025-12-05 09:50:30.33][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 09:50:30.37][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 09:50:41.31][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  10.

In [17]:
within_group

{1: {(0, 1): {'independent_variables': 4,
   'control_variables': 3,
   'response_variables': 4,
   'model_specification': 3,
   'conclusions': 3,
   'overall_similarity': 2},
  (0, 2): {'independent_variables': 5,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 4,
   'conclusions': 5,
   'overall_similarity': 5},
  (1, 2): {'independent_variables': 5,
   'control_variables': 3,
   'response_variables': 5,
   'model_specification': 4,
   'conclusions': 5,
   'overall_similarity': 4}},
 2: {(0, 1): {'independent_variables': 4,
   'control_variables': 3,
   'response_variables': 4,
   'model_specification': 4,
   'conclusions': 3,
   'overall_similarity': 2},
  (0, 2): {'independent_variables': 4,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 5,
   'conclusions': 5,
   'overall_similarity': 5},
  (1, 2): {'independent_variables': 5,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 4,
   'c

In [23]:
within_group[3]

{(0, 1): {'independent_variables': 4,
  'control_variables': 4,
  'response_variables': 4,
  'model_specification': 3,
  'conclusions': 3,
  'overall_similarity': 2},
 (0, 2): {'independent_variables': 4,
  'control_variables': 4,
  'response_variables': 3,
  'model_specification': 3,
  'conclusions': 5,
  'overall_similarity': 4},
 (1, 2): {'independent_variables': 5,
  'control_variables': 4,
  'response_variables': 4,
  'model_specification': 3,
  'conclusions': 5,
  'overall_similarity': 4}}

In [18]:
between_group

{(1,
  2): {(0, 0): {'independent_variables': 5,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 4,
   'conclusions': 5,
   'overall_similarity': 5}, (0, 1): {'independent_variables': 4,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 4,
   'conclusions': 4,
   'overall_similarity': 4}, (0, 2): {'independent_variables': 5,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 5,
   'conclusions': 5,
   'overall_similarity': 5}, (1, 0): {'independent_variables': 5,
   'control_variables': 4,
   'response_variables': 4,
   'model_specification': 4,
   'conclusions': 4,
   'overall_similarity': 4}, (1, 1): {'independent_variables': 4,
   'control_variables': 4,
   'response_variables': 4,
   'model_specification': 4,
   'conclusions': 5,
   'overall_similarity': 4}, (1, 2): {'independent_variables': 4,
   'control_variables': 3,
   'response_variables': 5,
   'model_specification': 4,
   'conclusi

In [19]:
# get average similarity score for each subcategory within each group
average_within_group = {}
for group_id, comparisons in within_group.items():
    category_sums = {
        "independent_variables": 0,
        "control_variables": 0,
        "response_variables": 0,
        "model_specification": 0,
        "conclusions": 0,
        "overall_similarity": 0
    }
    num_comparisons = len(comparisons)
    
    for comparison, scores in comparisons.items():
        for category, score in scores.items():
            category_sums[category] += score
    
    average_scores = {category: total / num_comparisons
                      for category, total in category_sums.items()}
    average_within_group[group_id] = average_scores

In [20]:
# show average within group rounded to nearest tenth
average_within_group

{1: {'independent_variables': 4.666666666666667,
  'control_variables': 3.3333333333333335,
  'response_variables': 4.666666666666667,
  'model_specification': 3.6666666666666665,
  'conclusions': 4.333333333333333,
  'overall_similarity': 3.6666666666666665},
 2: {'independent_variables': 4.333333333333333,
  'control_variables': 3.6666666666666665,
  'response_variables': 4.666666666666667,
  'model_specification': 4.333333333333333,
  'conclusions': 4.333333333333333,
  'overall_similarity': 4.0},
 3: {'independent_variables': 4.333333333333333,
  'control_variables': 4.0,
  'response_variables': 3.6666666666666665,
  'model_specification': 3.0,
  'conclusions': 4.333333333333333,
  'overall_similarity': 3.3333333333333335}}

In [21]:
# get average similarity score for each subcategory between each group
average_between_group = {}
for group_pair, comparisons in between_group.items():
    category_sums = {
        "independent_variables": 0,
        "control_variables": 0,
        "response_variables": 0,
        "model_specification": 0,
        "conclusions": 0,
        "overall_similarity": 0
    }
    num_comparisons = len(comparisons)
    
    for comparison, scores in comparisons.items():
        for category, score in scores.items():
            category_sums[category] += score
    
    average_scores = {category: total / num_comparisons
                      for category, total in category_sums.items()}
    average_between_group[group_pair] = average_scores

In [22]:
average_between_group

{(1, 2): {'independent_variables': 4.555555555555555,
  'control_variables': 4.0,
  'response_variables': 4.777777777777778,
  'model_specification': 4.111111111111111,
  'conclusions': 4.666666666666667,
  'overall_similarity': 4.444444444444445},
 (1, 3): {'independent_variables': 3.5555555555555554,
  'control_variables': 2.7777777777777777,
  'response_variables': 2.4444444444444446,
  'model_specification': 2.7777777777777777,
  'conclusions': 3.6666666666666665,
  'overall_similarity': 2.2222222222222223},
 (2, 3): {'independent_variables': 3.5555555555555554,
  'control_variables': 3.4444444444444446,
  'response_variables': 3.3333333333333335,
  'model_specification': 3.111111111111111,
  'conclusions': 3.888888888888889,
  'overall_similarity': 2.5555555555555554}}

In [26]:
features_3

{0: {'independent_variables': [{'description': 'Continuous femininity score of the hurricane name (higher = more feminine). This is a coder-rated masculinity-femininity index taken from the original data; we standardize it (z-score) for modeling.',
    'columns': ['femininity_z'],
    'transform_code': ["# Standardize (z-score). If constant or all NaN, fill with 0.0 to preserve observations.\nfert_std = df['femininity'].std(ddof=0)\nfert_mean = df['femininity'].mean()\nif pd.isna(fert_std) or fert_std == 0:\n    # No variation or all missing: impute neutral standardized score 0.0 for all rows\n    df['femininity_z'] = 0.0\nelse:\n    df['femininity_z'] = (df['femininity'] - fert_mean) / fert_std\n    # For any remaining NaNs (e.g., entries where femininity was missing), fill with 0.0\n    df['femininity_z'] = df['femininity_z'].fillna(0.0)"]}],
  'control_variables': [{'description': 'Binary indicator for whether the hurricane name was formally classified as a female name in the datase